# Comparison of Numba Implementations

## Import Libraries

In [1]:
import numpy as np
from matplotlib import pyplot as plt
import time, statistics
from numba import njit

## Numba Functions

In [5]:
# numba-hybrid
@njit
def compute_numba_hybrid_mandelbrot_point(max_iterations, c, bound):
    z = 0j
    for n in range(max_iterations):
        if z.real*z.real + z.imag*z.imag > bound*bound:
            return n
        z = z*z + c
    return max_iterations

def compute_numba_hybrid_mandelbrot(x_region, y_region, max_iterations, bound, power, res):
    result = np.zeros((res,res), dtype=np.int32)
    
    for i in range(res):
        for j in range(res):
            c = x_region[j] + y_region[i] * 1j
            result[i,j] = compute_numba_hybrid_mandelbrot_point(max_iterations, c, bound)
    
    return result

# numba-naive
@njit
def compute_numba_naive_mandelbrot(x_region, y_region, max_iterations, bound, power, res):
    result = np.zeros((res,res), dtype=np.int32)

    for i in range(res):
        for j in range(res):
            c = x_region[j] + y_region[i] * 1j
            z = 0j
            n = 0

            while n < max_iterations and z.real*z.real + z.imag*z.imag <= bound*bound:
                z = z*z + c
                n += 1
            result[i,j] = n

    return result

## Benchmark Function & Parameters

In [10]:
def benchmark_numba(func, *args):
    test_times = []

    for i in range(5): # 5 runs
        start_time = time.perf_counter()
        result = func(*args)
        test_time = time.perf_counter() - start_time
        test_times.append(test_time)

    median_time = statistics.median(test_times)

    return median_time

# regions
x_min, x_max = -2, 1
y_min, y_max = -1.5, 1.5

res = 1024

x_region = np.linspace(x_min, x_max, res)
y_region = np.linspace(y_min, y_max, res)

max_iterations = 100
bound = 2
power = 2

## Compare Numba Implementations

In [12]:
# test time of computation
warm_up = benchmark_numba(compute_numba_hybrid_mandelbrot, x_region, y_region, 1, bound, power, res)
hybrid_time = benchmark_numba(compute_numba_hybrid_mandelbrot, x_region, y_region, max_iterations, bound, power, res)
print(f'Median Compute Time for Hybrid Numba: {hybrid_time:.4f} seconds!')

warm_up = benchmark_numba(compute_numba_naive_mandelbrot, x_region, y_region, 1, bound, power, res)
naive_time = benchmark_numba(compute_numba_naive_mandelbrot, x_region, y_region, max_iterations, bound, power, res)
print(f'Median Compute Time for Naive Numba: {naive_time:.4f} seconds!')

ratio = hybrid_time / naive_time
print(f'Ratio: {ratio:.2f}x')

Median Compute Time for Hybrid Numba: 1.4242 seconds!
Median Compute Time for Naive Numba: 0.0491 seconds!
Ratio: 29.01x
